
## **Laboratorium 4. Fabryka konfiguracji**

W poprzednich laboratoriach konfiguracja była wczytywana z pliku YAML, iterowana, spłaszczana i mapowana na obiekty dataclass. W tym laboratorium rozwijamy ten mechanizm w kierunku programowania obiektowego: wydzielamy odpowiedzialności do klas, wprowadzamy fabrykę konfiguracji, a także przygotowujemy kod tak, aby był łatwy do rozbudowy o nowe typy sekcji i nowe źródła konfiguracji.

## Poziom podstawowy – Obiektowa reprezentacja sekcji konfiguracji

# **Założenia**

Program powinien:

* wczytywać konfigurację YAML,
* parsować wybrane sekcje do obiektów,
* wykorzystywać klasy reprezentujące poszczególne sekcje konfiguracji,
* spełniać implementacje niżej wymienonych metod.

# **Wymagania**

Utwórz data-klasy reprezentujące **wszystkie** sekcje konfiguracji:

* `AppConfig`
* `ServerConfig`
* `DatabaseConfig`

(Na ten moment proszę odłożyć na bok klasę do zebrania w.w. klas)

Każda klasa powinna:

* przechowywać dane sekcji,
* posiadać metodę validate() ewentualnie rzucającą wyjątek (nic nie zwraca),
* posiadać implementację metody __str__() dla ustawienia formatowania `str(dataobiekt)`
* posiadać metodę display() do wyświetlenia zawartośći klasy (nic nie zwraca).


In [ ]:
from dataclasses import dataclass, fields
import yaml
from pathlib import Path

class BaseConfigSection: #pusta klasa bazowa
    pass

#konfiguracja aplikacji
@dataclass(slots=True, frozen=True)
class AppConfig(BaseConfigSection):
    name: str
    debug: bool

    def validate(self) -> None:
        if not isinstance(self.name, str):
            raise TypeError("Pole name musi być typu string")
        if not isinstance(self.debug, bool):
            raise TypeError("Pole debug musi być typu bool")

        if not self.name:
            raise ValueError("Pole name nie może być puste")
        if len(self.name) < 3:
            raise ValueError("Pole name musi mieć co najmniej 3 znaki")
        if not self.name.isalpha():
            raise ValueError("Pole musi sie skladac z samych liter ")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"AppConfig: {str(self)}")

#konfiguracja serwera
@dataclass(slots=True, frozen=True)
class ServerConfig(BaseConfigSection):
    host: str
    port: int
    timeout: int 

    def validate(self) -> None:
        if not isinstance(self.host, str):
            raise TypeError("host musi być string")
        if not isinstance(self.port, int):
            raise TypeError("port musi byc integerem")
        if not isinstance(self.timeout, int):
            raise TypeError("timeout musi byc integerem")

        if not self.host:
            raise ValueError("host nie moze byc pusty")
        if not (1 <= self.port <= 65535):
            raise ValueError("port musi byc pomiedzy 1 a 65535")
        if self.timeout <= 0:
            raise ValueError("timeout musi byc > 0")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"ServerConfig: {str(self)}")

#konfiguracja bazy danych
@dataclass(slots=True, frozen=True)
class DatabaseConfig(BaseConfigSection):
    db_name: str
    user: str

    def validate(self) -> None:
        if not isinstance(self.db_name, str):
            raise TypeError("db_name musi byc stringiem")
        if not isinstance(self.user, str):
            raise TypeError("user musi byc stringiem")

        if not self.db_name:
            raise ValueError("db_name nie moze byc pusty")
        if not self.user:
            raise ValueError("user nie moze byc pusty")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"DatabaseConfig: {str(self)}")
            
if __name__ == "__main__":
    app = AppConfig(name="MojaAplikacja", debug = True)
    app.validate()
    app.display()

    server = ServerConfig(host="127.0.0.1", port = 8000, timeout=30)
    server.validate()
    server.display()

    db = DatabaseConfig(db_name="produkcja_db", user = "admin")
    db.validate()
    db.display()

    #plik yaml
    current_dir = Path(__file__).parent
    config_path = current_dir / "config.yaml"

    try:
        with open(config_path, "r", encoding="utf-8") as file:
            config_data = yaml.safe_load(file)
    except FileNotFoundError:
        print(f"Nie znaleziono pliku config.yaml w ścieżce: {config_path}")
        exit(1)
    print(f"wczytane dane z pliku yaml: {config_path}")
    print(config_data)
    print("-"*30)

    app = AppConfig(**config_data["app"]) # config_data["app"] to słownik: {'name': 'MojaAplikacja', 'debug': True}
    app.validate()
    app.display()

    server = ServerConfig(**config_data['server'])
    server.validate()
    server.display()
    
    db = DatabaseConfig(**config_data['database'])
    db.validate()
    db.display()
    




    
    

# **Efekt końcowy**

Program powinien:

* utworzyć obiekty dla wszystkich sekcji konfiguracji,
* wywołać ich metody walidujące,
* wyświetlić dane w czytelnej formie.

Przykład:

```python
app = AppConfig("MojaAplikacja", True)
app.validate()
app.display()
```



# **Poziom średniozaawansowany – Fabryka konfiguracji**

Wprowadź wzorzec projektowy [Factory](https://refactoring.guru/design-patterns/factory-method) ([przykład](https://refactoring.guru/design-patterns/factory-method/python/example)), który będzie odpowiadał za tworzenie odpowiednich obiektów konfiguracji na podstawie danych wejściowych.

# Założenia

Program powinien realizować wszystko z poziomu łatwego, a dodatkowo:

* zawierać klasę `ConfigFactory`,
* fabryka powinna tworzyć odpowiedni obiekt sekcji na podstawie nazwy sekcji,
* logika tworzenia obiektów nie powinna znajdować się w kodzie głównym.

# Wymagania
Utwórz klasę:


```python
from enum import StrEnum

class SectionType(StrEnum):
    APP = "app"
    SERVER = "server"
    DATABASE = "database"

class ConfigFactory:

    @staticmethod
    def create_section(section_name: str, data: dict) -> BaseConfig:
        section = SectionType(section_name.lower())
        match section:
            case SectionType.APP:
                return AppConfig(**data)
            case SectionType.SERVER:
                return # TODO ...
            case SectionType.DATABASE:
                return # TODO ...
            case _:
                raise # TODO ...
```

# Fabryka powinna:

* dla "app" zwrócić obiekt AppConfig,
* dla "server" zwrócić obiekt ServerConfig,
* dla "database" zwrócić obiekt DatabaseConfig,
* dla nieznanej sekcji zgłosić wyjątek, np. ValueError.



## Przykład użycia:

```python
loaded_server_yaml = {
    "host": "127.0.0.1",
    "port": 8080,
    "timeout": 30
}
section_obj = ConfigFactory.create_section("server", loaded_server_yaml)
section_obj.display()
```

# Dodatkowe założenie

Program powinien iterować po wszystkich sekcjach YAML i dla każdej sekcji wywoływać metodę fabryki.

In [ ]:
from dataclasses import dataclass, fields
import yaml
from pathlib import Path
from enum import StrEnum

class SectionType(StrEnum):
    APP = "app"
    SERVER = "server"
    DATABASE = "database"

class BaseConfigSection: #pusta klasa bazowa
    pass

class ConfigFactory:
    @staticmethod
    def create_section(section_name: str, data: dict) -> BaseConfigSection:
        try:
            section = SectionType(section_name.lower())
        except ValueError:
            raise ValueError(f"nieznana sekcja konfiguracji: '{section_name}'")

        match section:
            case SectionType.APP:
                return AppConfig(**data)
            case SectionType.SERVER:
                return ServerConfig(**data)
            case SectionType.DATABASE:
                return DatabaseConfig(**data)
            case _:
                raise NotImplementedError(f"brak implementacji dla sekcji: {section}")

#konfiguracja aplikacji
@dataclass(slots=True, frozen=True)
class AppConfig(BaseConfigSection):
    name: str
    debug: bool

    def validate(self) -> None:
        if not isinstance(self.name, str):
            raise TypeError("Pole name musi być typu string")
        if not isinstance(self.debug, bool):
            raise TypeError("Pole debug musi być typu bool")

        if not self.name:
            raise ValueError("Pole name nie może być puste")
        if len(self.name) < 3:
            raise ValueError("Pole name musi mieć co najmniej 3 znaki")
        if not self.name.isalpha():
            raise ValueError("Pole musi sie skladac z samych liter ")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"AppConfig: {str(self)}")

#konfiguracja serwera
@dataclass(slots=True, frozen=True)
class ServerConfig(BaseConfigSection):
    host: str
    port: int
    timeout: int 

    def validate(self) -> None:
        if not isinstance(self.host, str):
            raise TypeError("host musi być string")
        if not isinstance(self.port, int):
            raise TypeError("port musi byc integerem")
        if not isinstance(self.timeout, int):
            raise TypeError("timeout musi byc integerem")

        if not self.host:
            raise ValueError("host nie moze byc pusty")
        if not (1 <= self.port <= 65535):
            raise ValueError("port musi byc pomiedzy 1 a 65535")
        if self.timeout <= 0:
            raise ValueError("timeout musi byc > 0")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"ServerConfig: {str(self)}")

#konfiguracja bazy danych
@dataclass(slots=True, frozen=True)
class DatabaseConfig(BaseConfigSection):
    db_name: str
    user: str

    def validate(self) -> None:
        if not isinstance(self.db_name, str):
            raise TypeError("db_name musi byc stringiem")
        if not isinstance(self.user, str):
            raise TypeError("user musi byc stringiem")

        if not self.db_name:
            raise ValueError("db_name nie moze byc pusty")
        if not self.user:
            raise ValueError("user nie moze byc pusty")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"DatabaseConfig: {str(self)}")
            
if __name__ == "__main__":
    #plik yaml
    current_dir = Path(__file__).parent
    config_path = current_dir / "config.yaml"

    try:
        with open(config_path, "r", encoding="utf-8") as file:
            config_data = yaml.safe_load(file)
    except FileNotFoundError:
        print(f"Nie znaleziono pliku config.yaml w ścieżce: {config_path}")
        exit(1)

    print(f"wczytane dane z pliku yaml: {config_path}")
    print(config_data)
    print("-"*30)

    objects = {}

    for section_name, section_data in config_data.items():

        obj = ConfigFactory.create_section(section_name ,section_data)

        #walidacja i wyświetlanie
        obj.validate()
        obj.display()

        #zapisanie do słownika obiektów
        objects[section_name] = obj

    print("-"*30)
    print("Końcowy słownik gotowych obiektów:")
    print(objects)

    





    
    

# **Efekt końcowy**

Program powinien budować słownik gotowych obiektów konfiguracyjnych bez ręcznego if/else w kodzie głównym.

# **Poziom zaawansowany – Hierarchia klas i wspólny interfejs sekcji**

Zaprojektuj wspólną architekturę obiektową dla wszystkich sekcji konfiguracji, wykorzystując dziedziczenie lub klasy abstrakcyjne.

*(W tym miejscu wracamy do klasy zbiorczej, dla wszystkich dataklas)*

# Założenia

Program powinien realizować wszystko z poprzednich poziomów, a dodatkowo:

* zawierać klasę bazową dla sekcji konfiguracji (wyciągniecie wspólnej funkcjonalności),
* wymuszać wspólny interfejs dla wszystkich typów sekcji,
* umożliwiać łatwe dodanie nowych sekcji bez modyfikowania kodu głównego.

# Wymagania

W dotychczas pustej klasie abstrakcyjnej wprowadź następujące zmiany:
1. Dziedzicz po klasie `ABC`
2. Określ interfejs (listę metod), które muszą zostać zaimplementowane w każdej klasie dziedziczącej przy pomocy dekoratorów `@abstractmethod`

Przykład:

```python
from abc import ABC, abstractmethod

class BaseConfigSection(ABC):
    @abstractmethod
    def validate(self):
        pass

    @abstractmethod
    def display(self):
        pass
```

# Następnie:

* AppConfig, ServerConfig, DatabaseConfig powinny dziedziczyć po BaseConfigSection,
* każda klasa musi implementować validate() i display().

# Dodatkowo utwórz klasę nadrzędną, np. ApplicationConfig, która:

* przechowuje wszystkie sekcje jako obiekty,
* umożliwia wyświetlenie pełnej konfiguracji,
* posiada metodę validate_all().

In [ ]:
from abc import ABC , abstractmethod
from dataclasses import dataclass, fields
from enum import StrEnum
import yaml
from pathlib import Path


class SectionType(StrEnum):
    APP = "app"
    SERVER = "server"
    DATABASE = "database"

class BaseConfigSection(ABC): #Abstract Base Class

    #klasa abstrakcyjna wymuszająca interfejs dla kazdej sekcji konfiguracji
    @abstractmethod
    def validate(self) -> None:
        """każda sekcja musi implementować walidacje"""
        pass

    @abstractmethod
    def display(self) -> None:
        """każda sekcja musi umieć się wyświetlić"""
        pass

#fabryka
class ConfigFactory:
    @staticmethod
    def create_section(section_name: str, data: dict) -> BaseConfigSection:
        try:
            section = SectionType(section_name.lower())
        except ValueError:
            raise ValueError(f"nieznana sekcja konfiguracji: '{section_name}'")

        match section:
            case SectionType.APP:
                return AppConfig(**data)
            case SectionType.SERVER:
                return ServerConfig(**data)
            case SectionType.DATABASE:
                return DatabaseConfig(**data)
            case _:
                raise NotImplementedError(f"brak implementacji dla sekcji: {section}")


#konfiguracja aplikacji
@dataclass(slots=True, frozen=True)
class AppConfig(BaseConfigSection):
    name: str
    debug: bool

    def validate(self) -> None:
        if not isinstance(self.name, str):
            raise TypeError("Pole name musi być typu string")
        if not isinstance(self.debug, bool):
            raise TypeError("Pole debug musi być typu bool")

        if not self.name:
            raise ValueError("Pole name nie może być puste")
        if len(self.name) < 3:
            raise ValueError("Pole name musi mieć co najmniej 3 znaki")
        if not self.name.isalpha():
            raise ValueError("Pole musi sie skladac z samych liter ")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"AppConfig: {str(self)}")

#konfiguracja serwera
@dataclass(slots=True, frozen=True)
class ServerConfig(BaseConfigSection):
    host: str
    port: int
    timeout: int 

    def validate(self) -> None:
        if not isinstance(self.host, str):
            raise TypeError("host musi być string")
        if not isinstance(self.port, int):
            raise TypeError("port musi byc integerem")
        if not isinstance(self.timeout, int):
            raise TypeError("timeout musi byc integerem")

        if not self.host:
            raise ValueError("host nie moze byc pusty")
        if not (1 <= self.port <= 65535):
            raise ValueError("port musi byc pomiedzy 1 a 65535")
        if self.timeout <= 0:
            raise ValueError("timeout musi byc > 0")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"ServerConfig: {str(self)}")

#konfiguracja bazy danych
@dataclass(slots=True, frozen=True)
class DatabaseConfig(BaseConfigSection):
    db_name: str
    user: str

    def validate(self) -> None:
        if not isinstance(self.db_name, str):
            raise TypeError("db_name musi byc stringiem")
        if not isinstance(self.user, str):
            raise TypeError("user musi byc stringiem")

        if not self.db_name:
            raise ValueError("db_name nie moze byc pusty")
        if not self.user:
            raise ValueError("user nie moze byc pusty")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"DatabaseConfig: {str(self)}")
    
#klasa zbiorcza
@dataclass(slots=True, frozen=False)
class ApplicationConfig:
    sections: dict[str, BaseConfigSection]

    def validate_all(self) -> None:
        print("rozpoczynam globalną walidacje...")
        for section in self.sections.values():
            section.validate()
        print("walidacja zakonczona sukcesem")
    
    def display_all(self) -> None:
        print("\n---pełna konfiguracja aplikacji---")
        for name, section in self.sections.items():
            print(f"[{name.upper()}]")
            section.display()


#główny program       
if __name__ == "__main__":
    #plik yaml
    current_dir = Path(__file__).parent
    config_path = current_dir / "config.yaml"

    try:
        with open(config_path, "r", encoding="utf-8") as file:
            config_data = yaml.safe_load(file)
    except FileNotFoundError:
        print(f"Nie znaleziono pliku config.yaml w ścieżce: {config_path}")
        exit(1)

    #tworzenie obiektów przez fabryke
    parsed_sections = {}
    for section_name, section_data in config_data.items():
        obj = ConfigFactory.create_section(section_name, section_data)
        parsed_sections[section_name] = obj

    #utworzenie głównego obiektu aplikacji
    app_config = ApplicationConfig(sections=parsed_sections)

    #zwalidowanie całości
    app_config.validate_all()

    #wyświetlanie całości
    app_config.display_all()    




# Efekt końcowy

Po uruchomieniu program powinien:

* utworzyć pełny obiekt konfiguracji aplikacji,
* zwalidować wszystkie sekcje,
* wypisać całą konfigurację,
* działać zgodnie z zasadami programowania obiektowego: hermetyzacja, podział odpowiedzialności, rozszerzalność.

# **Zadanie dodatkowe – Rejestr fabryk i dynamiczne rozszerzanie systemu**

Rozszerz fabrykę tak, aby można było dynamicznie rejestrować nowe typy sekcji bez modyfikowania metody create_section().

# Założenia

Zamiast pisać wiele instrukcji 'if-elif' / 'match-case', stwórz mechanizm rejestracji klas w fabryce.

# Wymagania

Fabryka powinna posiadać rejestr klas:

In [ ]:
from dataclasses import dataclass, fields
from abc import ABC, abstractmethod

#interfejs
class BaseConfigSection(ABC):
    @abstractmethod
    def validate(self) -> None:
        pass

    @abstractmethod
    def display(self) -> None:
        pass

#dynamiczna fabryka
class ConfigFactory:
    _registry = {}

    @classmethod
    def register(cls, section_name: str, section_class):
        """metoda pozwalajaca zapisac nowa klase do rejestru"""
        cls._registry[section_name] = section_class

    @classmethod
    def create_section(cls, section_name: str, data: dict) -> BaseConfigSection:
        """tworzy obiekt szukajac odpowiedniej klasy w rejestrze"""
        if section_name not in cls._registry:
            raise ValueError(f"Nie ma zdefiniowanej konfiguracji dla: {section_name}")
        
        return cls._registry[section_name](**data)


#konfiguracja aplikacji
@dataclass(slots=True, frozen=True)
class AppConfig(BaseConfigSection):
    name: str
    debug: bool

    def validate(self) -> None:
        if not isinstance(self.name, str):
            raise TypeError("Pole name musi być typu string")
        if not isinstance(self.debug, bool):
            raise TypeError("Pole debug musi być typu bool")

        if not self.name:
            raise ValueError("Pole name nie może być puste")
        if len(self.name) < 3:
            raise ValueError("Pole name musi mieć co najmniej 3 znaki")
        if not self.name.isalpha():
            raise ValueError("Pole musi sie skladac z samych liter ")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"AppConfig: {str(self)}")

#konfiguracja serwera
@dataclass(slots=True, frozen=True)
class ServerConfig(BaseConfigSection):
    host: str
    port: int
    timeout: int 

    def validate(self) -> None:
        if not isinstance(self.host, str):
            raise TypeError("host musi być string")
        if not isinstance(self.port, int):
            raise TypeError("port musi byc integerem")
        if not isinstance(self.timeout, int):
            raise TypeError("timeout musi byc integerem")

        if not self.host:
            raise ValueError("host nie moze byc pusty")
        if not (1 <= self.port <= 65535):
            raise ValueError("port musi byc pomiedzy 1 a 65535")
        if self.timeout <= 0:
            raise ValueError("timeout musi byc > 0")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"ServerConfig: {str(self)}")

#konfiguracja bazy danych
@dataclass(slots=True, frozen=True)
class DatabaseConfig(BaseConfigSection):
    db_name: str
    user: str

    def validate(self) -> None:
        if not isinstance(self.db_name, str):
            raise TypeError("db_name musi byc stringiem")
        if not isinstance(self.user, str):
            raise TypeError("user musi byc stringiem")

        if not self.db_name:
            raise ValueError("db_name nie moze byc pusty")
        if not self.user:
            raise ValueError("user nie moze byc pusty")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"DatabaseConfig: {str(self)}")
    
#nowa sekcja
@dataclass(slots=True, frozen=True)
class LoggingConfig(BaseConfigSection):
    level: str
    file: str

    def validate(self) -> None:
        if self.level not in ["INFO", "DEBUG", "ERROR"]:
            raise ValueError("Niepoprawny poziom logowania")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )
    
    def display(self) -> None:
        print(f"LoggingConfig: {str(self)}")

#klasa zbiorcza
@dataclass(slots=True, frozen=False)
class ApplicationConfig:
    sections: dict[str, BaseConfigSection]

    def validate_all(self) -> None:
        print("rozpoczynam globalną walidacje...")
        for section in self.sections.values():
            section.validate()
        print("walidacja zakonczona sukcesem")
    
    def display_all(self) -> None:
        print("\n---pełna konfiguracja aplikacji---")
        for name, section in self.sections.items():
            print(f"[{name.upper()}]")
            section.display()

#główny program
if __name__ == "__main__":
    
    ConfigFactory.register("app", AppConfig)
    ConfigFactory.register("server", ServerConfig)
    ConfigFactory.register("database", DatabaseConfig)
    ConfigFactory.register("logging", LoggingConfig)

    config_data = {
        'app': {'name': 'MojaAplikacja', 'debug': True}, 
        'server': {'host': '127.0.0.1', 'port': 8080, 'timeout': 30}, 
        'database': {'db_name': 'moja_baza', 'user': 'admin'},
        'logging': {'level': 'INFO', 'file': '/var/log/app.log'} #nowe
    }

    parsed_sections = {}
    for section_name, section_data in config_data.items():
        obj =  ConfigFactory.create_section(section_name, section_data)
        parsed_sections[section_name] = obj

    app_config = ApplicationConfig(sections=parsed_sections)
    app_config.validate_all()
    app_config.display_all()


Następnie dodaj nową sekcję, np. logging, bez zmiany kodu samej fabryki:

# Efekt końcowy

System powinien dawać możliwość:

* łatwego dodawania nowych sekcji,
* zachowania zgodności z OOP,
* unikania rozbudowanych instrukcji warunkowych.